# NB06 — Live Scan（webcam + monitor）
成個課程嘅高潮：用你部機嘅 webcam + 第二個芒做真。結構光掃描。
**冇硬件？** 冇問題——預設行 virtual 模式，成個 pipeline 照跑。
有硬件嘅話，將下面 `HARDWARE = False` 改做 `True`。

In [1]:
import sys, pathlib
# repo-root relative imports so the notebook runs from anywhere
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / 'edu' / 'sl_edu').exists()) \
       if not (pathlib.Path.cwd() / 'edu' / 'sl_edu').exists() else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'edu'))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 4)
DATA = ROOT / 'data'
HARDWARE = False  # flip to True with a webcam + second display

## 1. Virtual mode：成個 loop 行晒，只係冇光子出入

In [2]:
import time
from sl_edu import backends, decode, oracle, patterns

# 'projector' generates and displays; 'camera' replays the dataset
pats = patterns.generate(1920, 1080, shift_time=4, n_periods=32)
proj = backends.PyMonitorProjector(exposure_ms=20, virtual=not HARDWARE)
proj.set_patterns(pats)

cam = backends.PyOpenCvCamera(
    0 if HARDWARE else str(DATA / 'shiftGraycode' / '%d.bmp'))
assert cam.open()

proj.project()
cam.start()
time.sleep(0.3)  # let the loop settle
frames = cam.capture(9, timeout_s=10)
proj.stop(); cam.stop()

print(f'captured {len(frames)} frames while projector showed patterns')
print(f'projector was on pattern #{proj.current_pattern} when we stopped')

captured 9 frames while projector showed patterns
projector was on pattern #5 when we stopped


## 2. Decode live capture

In [3]:
wrapped = decode.wrapped_phase(frames[:4], 4)
conf = decode.confidence_map(frames[:4])
floor_map = decode.floor_map(frames[4:], conf, wrapped, 32, 5.0)
absolute = decode.unwrap(wrapped, floor_map, conf, 5.0)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plt.imshow(wrapped, cmap='twilight'); plt.title('wrapped')
plt.subplot(1, 2, 2); plt.imshow(absolute, cmap='inferno'); plt.title('absolute')
plt.show()
print('live decode OK — same code path as nb03')

live decode OK — same code path as nb03


/var/folders/fy/9dvfxzq5767cnwkhj7z8381m0000gn/T/ipykernel_82713/515542665.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. 真硬件 checklist（`HARDWARE=True` 前）
1. HDMI 插第二個芒，**mirror mode 關掉**（要 extended desktop）
2. 個 projector window 拖落第二芒再 fullscreen（未實現自動定位——改 `cv2.moveWindow`）
3. 環境光盡量暗（LCD 亮度 << DLP 燈）
4. 場景靜止（時間協議同步，~200ms/scan）
5. macOS：Terminal/IDE 要有相機權限（系統設定 → 私隱 → 相機）

**同步嘅本質**：projector 每張 ≥16.7ms，相機連續影，
`capture(9)` 攞最新 9 張。冇硬件握手——係「約定時間表」協議。
工業觸發係對講機；我哋係鬧鐘。慢 100 倍，但靜態場景冇分別。

下一課：同 C++ 對照——我哋寫得啱唔啱？